# 07 — Classification des erreurs et annotation humaine

Les catégories automatiques issues de l'exécution SQLite sont utiles pour le diagnostic, mais elles ne remplacent pas l'annotation humaine des erreurs sémantiques ou des jointures exécutables.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ERROR_DIR = ROOT / 'results' / 'error_classification'
def read(path):
    with path.open(encoding='utf-8') as stream:
        return json.load(stream)
files = sorted(ERROR_DIR.glob('execution_based_*.json'))
summary = []
for path in files:
    frame = pd.DataFrame(read(path))
    summary.append({'pipeline': path.stem[-1], 'n': len(frame), **frame['category'].value_counts().to_dict()})
pd.DataFrame(summary).fillna(0).set_index('pipeline')

In [ ]:
annotation = ROOT / 'results/error_classification/manual_annotation_agreement.json'
if annotation.exists():
    display(pd.Series(read(annotation)).to_frame('valeur'))
else:
    print('Accord humain pas encore calculé. Créez une feuille de 60 items, faites annoter A et B indépendamment, puis lancez error_annotation.py score.')

## Guide de codage

- `structural` : SQL invalide, table/colonne inexistante, syntaxe ou structure de jointure incorrecte ;
- `semantic` : SQL exécutable mais résultat incompatible avec la question ou l'evidence ;
- `correct`, `ambiguous`, `other` : cas correct, non tranchable, ou hors catégories.

Rapportez le taux d'accord brut, Cohen kappa et les désaccords, avec au moins 30 sorties annotées par les deux étudiants.